In [166]:
import pandas as pd
import numpy as np
import datetime
import requests
from bs4 import BeautifulSoup

## Valores das ações

In [105]:
url = 'https://www.google.com/finance/markets/most-active'

# headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36"}

response = requests.get(url)
soup = BeautifulSoup(response.text, 'html.parser')
items = soup.find('ul', {'class':'sbnBtf'}).find_all('div', {'class':'SxcTic'})

In [156]:
df = pd.DataFrame()

for item in items:
    date = datetime.datetime.now()
    symbol = item.find('div', {'class':'iLEcy'}).find('div', {'class':'COaKTb'}).text
    name = item.find('div', {'class':'ZvmM7'}).text
    price = item.find('div', {'class':'YMlKec'}).text
    change = item.find('span', {'class':'P2Luy'}).text

    tmp = pd.DataFrame(
        data = {
            'date':[date],
            'symbol':[symbol],
            'name':[name],
            'price':[price],
            'change':[change]
        }
    )

    df = pd.concat([df, tmp])

df['date'] = pd.to_datetime(df['date'], format = '%Y-%m-%d %H:%M:%S')

## Setores das empresas

In [167]:
setores_empresas = pd.read_excel("setores_empresas_b3.xlsx")
setores_empresas['setores_split'] = setores_empresas['Setor'].str.split(' / ')
setores_empresas['size'] = setores_empresas['setores_split'].apply(lambda x: len(x))

df = pd.DataFrame()
for i in [1, 2, 3]:
    tmp = setores_empresas[setores_empresas['size'] == i]
    for j in range(i):
        tmp[f'setor_n{j+1}'] = tmp['setores_split'].apply(lambda x: x[j])
    df = pd.concat([df, tmp])

df = df.drop(['Setor', 'setores_split', 'size'], axis = 1)
df.columns = df.columns.str.lower()
df['setor_n1'] = np.where(df['setor_n1'] == 'Financeiro', 'Financeiro e Outros', df['setor_n1'])
df

,código,ação,tipo,site,setor_n1,setor_n2,setor_n3
19,CSAN3,COSAN,ON NM,http://ri.cosan.com.br/,Petróleo. Gás e Biocombustíveis,NaN,NaN
48,PETR3,PETROBRAS,ON N2,http://www.petrobras.com.br/,Petróleo. Gás e Biocombustíveis,NaN,NaN
49,PETR4,PETROBRAS,PN N2,http://www.petrobras.com.br/,Petróleo. Gás e Biocombustíveis,NaN,NaN
59,UGPA3,ULTRAPAR,ON ED NM,http://www.ultra.com.br/,Petróleo. Gás e Biocombustíveis,NaN,NaN
15,CMIG4,CEMIG,PN N1,http://ri.cemig.com.br/,Utilidade Pública,Energia Elétrica,NaN
...,...,...,...,...,...,...,...
47,PCAR3,P.ACUCAR-CBD,ON ED NM,https://www.paodeacucar.com/,Consumo não Cíclico,Comércio e Distribuição,Alimentos
52,RAIL3,RUMO S.A.,ON NM,http://rumolog.com/,Bens Industriais,Transporte,Transporte Ferroviário
54,SANB11,SANTANDER BR,UNT,https://www.santander.com.br/,Financeiro e Outros,Intermediários Financeiros,Bancos
60,USIM5,USIMINAS,PNA N1,https://www.usiminas.com/,Materiais Básicos,Siderurgia e Metalurgia,Siderurgia


In [168]:
df['setor_n1'].value_counts()

setor_n1
Financeiro e Outros                13
Utilidade Pública                   9
Materiais Básicos                   9
Consumo Cíclico                     8
Consumo não Cíclico                 8
Bens Industriais                    7
Saúde                               5
Petróleo. Gás e Biocombustíveis     4
Tecnologia da Informação            1
Name: count, dtype: int64